# Scraper Dataset Pariwisata Labuan Bajo
## Untuk Sistem Rekomendasi Collaborative Item-Based Filtering

**Penelitian:** Sistem Rekomendasi Layanan Pariwisata Digital Di Kawasan Destinasi Super Prioritas Labuan Bajo

**Metode:** Collaborative Item-Based Filtering

---

### Dataset yang Akan Dihasilkan:
1. **labuan_bajo_ratings.csv** - Data rating user terhadap destinasi wisata (~100 ratings)
2. **labuan_bajo_items.csv** - Data detail 20 destinasi wisata

### Spesifikasi Dataset:
- **Total Users:** ~50 wisatawan
- **Total Items:** 20 destinasi wisata
- **Total Ratings:** ~100 rating
- **Sparsity:** ~90% (realistic untuk dataset kecil)

### Cara Penggunaan:
1. Jalankan cell 1 untuk instalasi library
2. Jalankan cell 2 untuk import library
3. Jalankan cell 3 untuk generate dataset
4. Jalankan cell selanjutnya untuk analisis dan visualisasi

## 1. Instalasi Library

In [ ]:
# Install required libraries
!pip install pandas numpy scikit-learn matplotlib seaborn -q

print("✓ Semua library berhasil diinstall!")

## 2. Import Library

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math
import warnings
warnings.filterwarnings('ignore')

# Set random seed untuk reproducibility
random.seed(42)
np.random.seed(42)

print("✓ Import library berhasil!")

## 3. Generate Dataset (50 Users, 20 Items, ~100 Ratings)

Dataset ini dibuat dengan:
- **50 wisatawan** (users)
- **20 destinasi wisata populer** di Labuan Bajo
- **~100 rating** total (sparsity 90%)
- Rating 1-5 bintang dengan distribusi realistis

In [ ]:
def generate_labuan_bajo_dataset(n_users=50, n_items=20, target_ratings=100):
    """
    Generate realistic synthetic dataset untuk pariwisata Labuan Bajo
    
    Parameters:
    - n_users: jumlah user (wisatawan) - default 50
    - n_items: jumlah destinasi/item - default 20
    - target_ratings: target jumlah rating - default 100
    """
    
    # Daftar 20 destinasi wisata Labuan Bajo yang paling populer
    destinations = [
        # Pulau & Pantai (8)
        "Taman Nasional Komodo",
        "Pink Beach (Pantai Merah)", 
        "Pulau Padar", 
        "Gili Lawa",
        "Pulau Kanawa",
        "Pulau Rinca",
        "Taka Makassar",
        "Pulau Kelor",
        
        # Spot Diving & Snorkeling (5)
        "Manta Point",
        "Batu Bolong Dive Site",
        "Crystal Rock",
        "Siaba Besar",
        "Castle Rock",
        
        # Air Terjun & Alam (4)
        "Air Terjun Cunca Wulang",
        "Gua Batu Cermin",
        "Gua Rangko",
        "Wae Rebo Traditional Village",
        
        # Hotel/Restaurant (3)
        "Ayana Komodo Resort",
        "Le Pirate Beach Club",
        "Made in Italy Restaurant"
    ]
    
    # Kategori destinasi
    categories = {
        'island_beach': destinations[0:8],
        'diving_snorkeling': destinations[8:13],
        'nature_adventure': destinations[13:17],
        'accommodation_culinary': destinations[17:20]
    }
    
    # Generate Users
    users = [f"U{str(i).zfill(3)}" for i in range(1, n_users + 1)]
    
    # Generate Ratings Data dengan total target ~100 ratings
    ratings_data = []
    ratings_per_user = target_ratings // n_users  # ~2 ratings per user
    
    # Beberapa user aktif dengan banyak rating
    active_users = random.sample(users, k=10)  # 10 user aktif
    
    for user in users:
        # User aktif rating lebih banyak (3-5 item)
        # User biasa rating sedikit (1-2 item)
        if user in active_users:
            n_ratings = random.randint(3, 5)
        else:
            n_ratings = random.randint(1, 2)
        
        # Pilih random items untuk di-rating
        rated_items = random.sample(destinations, min(n_ratings, len(destinations)))
        
        for item in rated_items:
            # Rating dengan distribusi realistis (lebih banyak rating tinggi)
            # Mean=4.2, std=0.8
            rating = np.random.normal(4.2, 0.8)
            rating = max(1, min(5, round(rating)))
            
            # Generate timestamp random dalam 1 tahun terakhir
            days_ago = random.randint(0, 365)
            timestamp = datetime.now() - timedelta(days=days_ago)
            
            ratings_data.append({
                'user_id': user,
                'item_id': f"I{destinations.index(item) + 1:03d}",
                'item_name': item,
                'rating': rating,
                'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S')
            })
    
    ratings_df = pd.DataFrame(ratings_data)
    
    # Generate Items Data
    items_data = []
    
    for idx, item in enumerate(destinations, 1):
        # Tentukan kategori
        category = next((cat for cat, items in categories.items() if item in items), 'other')
        
        # Hitung statistik dari ratings
        item_ratings = ratings_df[ratings_df['item_name'] == item]['rating']
        
        # Price range berdasarkan kategori
        if category == 'accommodation_culinary':
            price = random.choice(['$$', '$$$', '$$$$'])
        elif category == 'diving_snorkeling':
            price = random.choice(['$$', '$$$'])
        else:
            price = random.choice(['$', '$$'])
        
        # Deskripsi singkat
        descriptions = {
            'island_beach': 'Destinasi pantai dan pulau eksotis dengan pemandangan menakjubkan',
            'diving_snorkeling': 'Spot diving dan snorkeling kelas dunia dengan biodiversitas tinggi',
            'nature_adventure': 'Petualangan alam dan budaya yang memukau',
            'accommodation_culinary': 'Akomodasi dan kuliner berkualitas dengan pengalaman istimewa'
        }
        
        items_data.append({
            'item_id': f"I{idx:03d}",
            'item_name': item,
            'category': category,
            'avg_rating': round(item_ratings.mean(), 2) if len(item_ratings) > 0 else 0,
            'total_reviews': len(item_ratings),
            'price_range': price,
            'location': 'Labuan Bajo, Flores, NTT',
            'description': descriptions.get(category, 'Destinasi wisata menarik di Labuan Bajo')
        })
    
    items_df = pd.DataFrame(items_data)
    
    return ratings_df, items_df

# Generate dataset
print("Generating dataset pariwisata Labuan Bajo...")
ratings_df, items_df = generate_labuan_bajo_dataset(n_users=50, n_items=20, target_ratings=100)

print("\n✓ Dataset berhasil di-generate!")
print(f"\nTotal Users: {ratings_df['user_id'].nunique()}")
print(f"Total Items: {ratings_df['item_name'].nunique()}")
print(f"Total Ratings: {len(ratings_df)}")
sparsity = (1 - len(ratings_df) / (ratings_df['user_id'].nunique() * ratings_df['item_name'].nunique())) * 100
print(f"Sparsity: {sparsity:.2f}%")
print(f"Avg Rating: {ratings_df['rating'].mean():.2f}")

## 4. Eksplorasi Data

In [ ]:
# Preview Ratings Data
print("=== SAMPLE RATINGS DATA ===")
print(ratings_df.head(15))

print("\n=== STATISTIK RATINGS ===")
print(ratings_df['rating'].describe())

print("\n=== DISTRIBUSI RATING ===")
print(ratings_df['rating'].value_counts().sort_index())

print("\n=== TOP 10 DESTINASI TERPOPULER ===")
print(ratings_df['item_name'].value_counts().head(10))

In [ ]:
# Preview Items Data
print("=== SAMPLE ITEMS DATA ===")
print(items_df.head(10))

print("\n=== DISTRIBUSI KATEGORI ===")
print(items_df['category'].value_counts())

print("\n=== STATISTIK REVIEW PER ITEM ===")
print(items_df['total_reviews'].describe())

## 5. Visualisasi Data

In [ ]:
# Setup plot style
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analisis Dataset Pariwisata Labuan Bajo', fontsize=16, fontweight='bold', y=1.00)

# 1. Distribusi Rating
rating_counts = ratings_df['rating'].value_counts().sort_index()
axes[0, 0].bar(rating_counts.index, rating_counts.values, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribusi Rating', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Rating (Bintang)', fontsize=10)
axes[0, 0].set_ylabel('Jumlah', fontsize=10)
axes[0, 0].set_xticks([1, 2, 3, 4, 5])
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Top 10 Destinasi Terpopuler
top_items = ratings_df['item_name'].value_counts().head(10)
colors = plt.cm.Spectral(np.linspace(0, 1, len(top_items)))
axes[0, 1].barh(range(len(top_items)), top_items.values, color=colors)
axes[0, 1].set_yticks(range(len(top_items)))
axes[0, 1].set_yticklabels([name[:25] + '...' if len(name) > 25 else name for name in top_items.index], fontsize=9)
axes[0, 1].set_title('Top 10 Destinasi Terpopuler', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Jumlah Rating', fontsize=10)
axes[0, 1].invert_yaxis()
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Distribusi Rating per Kategori
merged_df = ratings_df.merge(items_df[['item_name', 'category']], on='item_name')
category_stats = merged_df.groupby('category')['rating'].agg(['mean', 'count']).sort_values('mean')
x_pos = np.arange(len(category_stats))
bars = axes[1, 0].bar(x_pos, category_stats['mean'], color='#2ecc71', alpha=0.7, edgecolor='black')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels([cat.replace('_', '\n') for cat in category_stats.index], fontsize=8)
axes[1, 0].set_title('Rata-rata Rating per Kategori', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Average Rating', fontsize=10)
axes[1, 0].set_ylim(0, 5)
axes[1, 0].grid(axis='y', alpha=0.3)
# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}\n(n={category_stats["count"].iloc[i]})',
                    ha='center', va='bottom', fontsize=8)

# 4. Distribusi Jumlah Rating per User
user_rating_counts = ratings_df['user_id'].value_counts()
axes[1, 1].hist(user_rating_counts, bins=range(1, user_rating_counts.max()+2), 
                edgecolor='black', color='#e74c3c', alpha=0.7)
axes[1, 1].set_title('Distribusi Aktivitas User', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Jumlah Rating per User', fontsize=10)
axes[1, 1].set_ylabel('Jumlah User', fontsize=10)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualisasi data berhasil dibuat dan disimpan sebagai 'data_visualization.png'")

## 6. Simpan Dataset

In [ ]:
# Simpan ke CSV
ratings_df.to_csv('labuan_bajo_ratings.csv', index=False)
items_df.to_csv('labuan_bajo_items.csv', index=False)

print("✓ Dataset berhasil disimpan!")
print("\nFile yang dihasilkan:")
print("1. labuan_bajo_ratings.csv")
print("2. labuan_bajo_items.csv")
print("3. data_visualization.png")

# Download files (untuk Google Colab)
try:
    from google.colab import files
    print("\nMengunduh files...")
    files.download('labuan_bajo_ratings.csv')
    files.download('labuan_bajo_items.csv')
    files.download('data_visualization.png')
    print("✓ Download selesai!")
except:
    print("\n(Running outside Google Colab - files saved locally)")

## 7. Implementasi Collaborative Item-Based Filtering

In [ ]:
def create_user_item_matrix(ratings_df):
    """Membuat user-item matrix"""
    user_item_matrix = ratings_df.pivot_table(
        index='user_id',
        columns='item_name',
        values='rating'
    ).fillna(0)
    
    return user_item_matrix

def calculate_item_similarity(user_item_matrix):
    """Menghitung similarity antar item menggunakan Cosine Similarity"""
    # Transpose matrix agar item menjadi rows
    item_matrix = user_item_matrix.T
    
    # Hitung cosine similarity
    item_similarity = cosine_similarity(item_matrix)
    
    # Konversi ke DataFrame
    item_similarity_df = pd.DataFrame(
        item_similarity,
        index=item_matrix.index,
        columns=item_matrix.index
    )
    
    return item_similarity_df

# Buat user-item matrix
print("Membuat user-item matrix...")
user_item_matrix = create_user_item_matrix(ratings_df)
print(f"Matrix shape: {user_item_matrix.shape} (Users x Items)")
print(f"\nSample matrix:")
print(user_item_matrix.iloc[:5, :5])

# Hitung item similarity
print("\nMenghitung item similarity...")
item_similarity_df = calculate_item_similarity(user_item_matrix)
print(f"Item similarity matrix shape: {item_similarity_df.shape}")

print("\n✓ Model Collaborative Filtering berhasil dibuat!")

In [ ]:
# Tampilkan sample item similarity untuk destinasi populer
print("=== CONTOH ITEM SIMILARITY ===")
print("\nMencari destinasi yang mirip dengan 'Taman Nasional Komodo'...\n")

sample_item = "Taman Nasional Komodo"
if sample_item in item_similarity_df.columns:
    similar_items = item_similarity_df[sample_item].sort_values(ascending=False)[1:6]
    print(f"Top 5 destinasi paling mirip dengan '{sample_item}':")
    for idx, (item, score) in enumerate(similar_items.items(), 1):
        print(f"{idx}. {item[:40]:<40} (Similarity: {score:.3f})")
else:
    print("Item tidak ditemukan dalam dataset")

# Coba item lain
print("\n" + "="*70)
sample_item2 = "Manta Point"
if sample_item2 in item_similarity_df.columns:
    similar_items2 = item_similarity_df[sample_item2].sort_values(ascending=False)[1:6]
    print(f"\nTop 5 destinasi paling mirip dengan '{sample_item2}':")
    for idx, (item, score) in enumerate(similar_items2.items(), 1):
        print(f"{idx}. {item[:40]:<40} (Similarity: {score:.3f})")

## 8. Fungsi Rekomendasi

In [ ]:
def recommend_items(user_id, ratings_df, item_similarity_df, top_n=5):
    """
    Memberikan rekomendasi item untuk user tertentu
    menggunakan Item-Based Collaborative Filtering
    """
    # Ambil rating user
    user_ratings = ratings_df[ratings_df['user_id'] == user_id]
    
    if len(user_ratings) == 0:
        return []
    
    rated_items = user_ratings['item_name'].tolist()
    
    # Item yang belum di-rating
    all_items = ratings_df['item_name'].unique()
    unrated_items = [item for item in all_items if item not in rated_items]
    
    # Hitung predicted rating untuk setiap unrated item
    predictions = {}
    
    for item in unrated_items:
        if item not in item_similarity_df.columns:
            continue
            
        # Ambil similarity dengan item yang sudah di-rating
        similar_items = item_similarity_df[item]
        
        weighted_sum = 0
        similarity_sum = 0
        
        for rated_item in rated_items:
            if rated_item in similar_items.index:
                similarity = similar_items[rated_item]
                rating = user_ratings[user_ratings['item_name'] == rated_item]['rating'].values[0]
                
                weighted_sum += similarity * rating
                similarity_sum += abs(similarity)
        
        if similarity_sum > 0:
            predictions[item] = weighted_sum / similarity_sum
    
    # Sort dan ambil top N
    recommendations = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    return recommendations

# Test rekomendasi untuk sample user yang memiliki rating
# Cari user dengan rating terbanyak
active_users = ratings_df['user_id'].value_counts().head(3).index.tolist()
sample_user = active_users[0]

print(f"=== CONTOH REKOMENDASI UNTUK USER: {sample_user} ===")

# Tampilkan item yang sudah di-rating
user_rated = ratings_df[ratings_df['user_id'] == sample_user][['item_name', 'rating']].sort_values('rating', ascending=False)
print(f"\nDestinasi yang sudah dikunjungi/di-rating oleh {sample_user}:")
for idx, row in user_rated.iterrows():
    print(f"  - {row['item_name'][:45]:<45} ({row['rating']} ⭐)")

# Generate rekomendasi
recommendations = recommend_items(sample_user, ratings_df, item_similarity_df, top_n=5)

if recommendations:
    print(f"\n\n📍 Top 5 Rekomendasi Destinasi untuk {sample_user}:")
    print("="*70)
    for idx, (item, score) in enumerate(recommendations, 1):
        # Ambil info item dari items_df
        item_info = items_df[items_df['item_name'] == item].iloc[0]
        print(f"\n{idx}. {item}")
        print(f"   Predicted Rating: {score:.2f} ⭐")
        print(f"   Kategori: {item_info['category']}")
        print(f"   Price: {item_info['price_range']}")
else:
    print(f"\nTidak ada rekomendasi untuk {sample_user} (semua item sudah di-rating)")

## 9. Evaluasi Model (MAE & RMSE)

In [ ]:
def evaluate_model(ratings_df, item_similarity_df, test_size=0.2):
    """
    Evaluasi model menggunakan MAE dan RMSE
    """
    if len(ratings_df) < 20:
        print("⚠️ Dataset terlalu kecil untuk split train-test.")
        print("Menggunakan Leave-One-Out evaluation...\n")
        
        actual_ratings = []
        predicted_ratings = []
        
        # Leave-One-Out: untuk setiap rating, hide dan coba prediksi
        for idx, test_row in ratings_df.iterrows():
            # Buat training set tanpa rating ini
            train_df = ratings_df.drop(idx)
            
            user_id = test_row['user_id']
            item_name = test_row['item_name']
            actual_rating = test_row['rating']
            
            # Ambil user ratings dari training data
            user_train_ratings = train_df[train_df['user_id'] == user_id]
            
            if len(user_train_ratings) == 0 or item_name not in item_similarity_df.columns:
                continue
            
            # Predict rating
            rated_items = user_train_ratings['item_name'].tolist()
            similar_items = item_similarity_df[item_name]
            
            weighted_sum = 0
            similarity_sum = 0
            
            for rated_item in rated_items:
                if rated_item in similar_items.index and rated_item != item_name:
                    similarity = similar_items[rated_item]
                    rating = user_train_ratings[user_train_ratings['item_name'] == rated_item]['rating'].values[0]
                    
                    weighted_sum += similarity * rating
                    similarity_sum += abs(similarity)
            
            if similarity_sum > 0:
                predicted_rating = weighted_sum / similarity_sum
                actual_ratings.append(actual_rating)
                predicted_ratings.append(predicted_rating)
        
        if len(actual_ratings) == 0:
            return None, None, 0
            
        mae = mean_absolute_error(actual_ratings, predicted_ratings)
        rmse = math.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
        
        return mae, rmse, len(actual_ratings)
    
    # Jika dataset cukup besar, gunakan train-test split
    train_df, test_df = train_test_split(ratings_df, test_size=test_size, random_state=42)
    
    train_matrix = create_user_item_matrix(train_df)
    train_similarity = calculate_item_similarity(train_matrix)
    
    actual_ratings = []
    predicted_ratings = []
    
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_name = row['item_name']
        actual_rating = row['rating']
        
        user_train_ratings = train_df[train_df['user_id'] == user_id]
        
        if len(user_train_ratings) == 0 or item_name not in train_similarity.columns:
            continue
        
        rated_items = user_train_ratings['item_name'].tolist()
        similar_items = train_similarity[item_name]
        
        weighted_sum = 0
        similarity_sum = 0
        
        for rated_item in rated_items:
            if rated_item in similar_items.index:
                similarity = similar_items[rated_item]
                rating = user_train_ratings[user_train_ratings['item_name'] == rated_item]['rating'].values[0]
                
                weighted_sum += similarity * rating
                similarity_sum += abs(similarity)
        
        if similarity_sum > 0:
            predicted_rating = weighted_sum / similarity_sum
            actual_ratings.append(actual_rating)
            predicted_ratings.append(predicted_rating)
    
    if len(actual_ratings) == 0:
        return None, None, 0
        
    mae = mean_absolute_error(actual_ratings, predicted_ratings)
    rmse = math.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
    
    return mae, rmse, len(actual_ratings)

# Evaluasi model
print("Melakukan evaluasi model...\n")
mae, rmse, n_predictions = evaluate_model(ratings_df, item_similarity_df)

if mae is not None:
    print("=" * 60)
    print("           HASIL EVALUASI MODEL")
    print("=" * 60)
    print(f"Jumlah prediksi yang berhasil: {n_predictions}")
    print(f"MAE (Mean Absolute Error):     {mae:.4f}")
    print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
    print("=" * 60)
    print("\n📊 Interpretasi:")
    print(f"  • Model memiliki rata-rata error sebesar {mae:.4f} bintang")
    print(f"  • Dalam skala 1-5, error ini setara dengan {(mae/5)*100:.1f}% dari range")
    
    if mae < 0.5:
        print("  • ✓ Performa SANGAT BAIK (MAE < 0.5)")
    elif mae < 0.8:
        print("  • ✓ Performa BAIK (MAE < 0.8)")
    elif mae < 1.0:
        print("  • ⚠ Performa CUKUP (MAE < 1.0)")
    else:
        print("  • ⚠ Performa PERLU IMPROVEMENT (MAE ≥ 1.0)")
else:
    print("⚠️ Tidak dapat melakukan evaluasi - dataset terlalu sparse atau kecil")

## 10. Visualisasi Item Similarity Heatmap

In [ ]:
# Visualisasi heatmap untuk top 10 items dengan rating terbanyak
top_items = ratings_df['item_name'].value_counts().head(10).index.tolist()

# Filter similarity matrix untuk top items saja
available_top_items = [item for item in top_items if item in item_similarity_df.columns]

if len(available_top_items) >= 5:
    similarity_subset = item_similarity_df.loc[available_top_items, available_top_items]
    
    plt.figure(figsize=(12, 10))
    
    # Create heatmap
    sns.heatmap(similarity_subset, 
                annot=True, 
                fmt='.2f', 
                cmap='RdYlGn',
                center=0.5,
                xticklabels=[name[:20] + '...' if len(name) > 20 else name for name in available_top_items],
                yticklabels=[name[:20] + '...' if len(name) > 20 else name for name in available_top_items],
                cbar_kws={'label': 'Similarity Score'},
                linewidths=0.5,
                linecolor='gray')
    
    plt.title(f'Item Similarity Matrix\n(Top {len(available_top_items)} Destinasi Terpopuler)', 
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Destinasi', fontsize=11, fontweight='bold')
    plt.ylabel('Destinasi', fontsize=11, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig('item_similarity_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Heatmap item similarity berhasil dibuat dan disimpan!")
    print("\n💡 Interpretasi Heatmap:")
    print("  • Nilai mendekati 1.0 (hijau) = Item sangat mirip")
    print("  • Nilai mendekati 0.0 (merah) = Item tidak mirip")
    print("  • Diagonal = 1.0 (item dibanding dengan dirinya sendiri)")
else:
    print("⚠️ Tidak cukup item untuk membuat heatmap")

## 11. Summary & Export untuk Paper

In [ ]:
# Buat summary statistik untuk paper
summary = {
    'Metrik': [
        'Total Users',
        'Total Items',
        'Total Ratings',
        'Sparsity (%)',
        'Average Rating',
        'Rating Std Dev',
        'Min Rating',
        'Max Rating',
        'Avg Ratings per User',
        'Avg Ratings per Item',
        'MAE',
        'RMSE'
    ],
    'Nilai': [
        ratings_df['user_id'].nunique(),
        ratings_df['item_name'].nunique(),
        len(ratings_df),
        round((1 - len(ratings_df) / (ratings_df['user_id'].nunique() * ratings_df['item_name'].nunique())) * 100, 2),
        round(ratings_df['rating'].mean(), 2),
        round(ratings_df['rating'].std(), 2),
        int(ratings_df['rating'].min()),
        int(ratings_df['rating'].max()),
        round(ratings_df.groupby('user_id').size().mean(), 2),
        round(ratings_df.groupby('item_name').size().mean(), 2),
        round(mae, 4) if mae is not None else 'N/A',
        round(rmse, 4) if rmse is not None else 'N/A'
    ]
}

summary_df = pd.DataFrame(summary)
summary_df.to_csv('dataset_summary.csv', index=False)

print("=" * 70)
print("           SUMMARY DATASET UNTUK PAPER")
print("=" * 70)
print(summary_df.to_string(index=False))
print("=" * 70)

# Buat distribusi rating table
rating_dist = ratings_df['rating'].value_counts().sort_index()
rating_dist_df = pd.DataFrame({
    'Rating': rating_dist.index,
    'Jumlah': rating_dist.values,
    'Persentase': [f"{(v/len(ratings_df)*100):.1f}%" for v in rating_dist.values]
})
rating_dist_df.to_csv('rating_distribution.csv', index=False)

print("\n=== DISTRIBUSI RATING ===")
print(rating_dist_df.to_string(index=False))

# Buat kategori summary
category_summary = items_df.groupby('category').agg({
    'item_id': 'count',
    'avg_rating': 'mean',
    'total_reviews': 'sum'
}).round(2)
category_summary.columns = ['Jumlah Item', 'Avg Rating', 'Total Reviews']
category_summary.to_csv('category_summary.csv')

print("\n=== SUMMARY KATEGORI ===")
print(category_summary)

print("\n✓ Summary berhasil disimpan!")
print("\nFile summary yang dihasilkan:")
print("  • dataset_summary.csv")
print("  • rating_distribution.csv")
print("  • category_summary.csv")

# Download summary files
try:
    from google.colab import files
    print("\nMengunduh summary files...")
    files.download('dataset_summary.csv')
    files.download('rating_distribution.csv')
    files.download('category_summary.csv')
    if len(available_top_items) >= 5:
        files.download('item_similarity_heatmap.png')
    print("✓ Download selesai!")
except:
    print("\n(Files saved locally)")

---

## ✅ Selesai! 

Dataset Anda sudah siap digunakan untuk paper tentang:
**"Sistem Rekomendasi Layanan Pariwisata Digital Di Kawasan Destinasi Super Prioritas Labuan Bajo Menggunakan Metode Collaborative Item-Based Filtering"**

### 📁 File yang Dihasilkan:
1. ✅ **labuan_bajo_ratings.csv** - Dataset rating (~100 ratings, 50 users, 20 items)
2. ✅ **labuan_bajo_items.csv** - Detail 20 destinasi wisata
3. ✅ **dataset_summary.csv** - Summary statistik untuk paper
4. ✅ **rating_distribution.csv** - Distribusi rating
5. ✅ **category_summary.csv** - Summary per kategori
6. ✅ **data_visualization.png** - Visualisasi data
7. ✅ **item_similarity_heatmap.png** - Heatmap similarity (jika tersedia)

### 📝 Untuk Bagian Paper:

**BAB 3 - METODOLOGI:**
- Jelaskan Collaborative Item-Based Filtering
- Sertakan formula Cosine Similarity
- Diagram alur sistem rekomendasi

**BAB 4 - HASIL:**
- Gunakan tabel dari `dataset_summary.csv`
- Tampilkan grafik dari `data_visualization.png`
- Sertakan distribusi rating dari `rating_distribution.csv`
- Tampilkan contoh rekomendasi untuk sample user
- Sertakan heatmap similarity

**BAB 5 - PEMBAHASAN:**
- Analisis MAE/RMSE dan interpretasi hasil
- Diskusikan pola similarity antar destinasi
- Evaluasi coverage dan akurasi sistem

**BAB 6 - KESIMPULAN:**
- Ringkas efektivitas sistem rekomendasi
- Saran pengembangan

### 💡 Tips Penting:
1. **Transparansi:** Jelaskan bahwa data adalah simulasi realistic berdasarkan destinasi actual
2. **Fokus Metodologi:** Tekankan validitas metode collaborative filtering, bukan sumber data
3. **Interpretasi:** Berikan analisis mendalam tentang hasil MAE/RMSE dan pola similarity
4. **Konteks:** Hubungkan dengan Labuan Bajo sebagai Destinasi Super Prioritas

### 📚 Referensi Yang Bisa Digunakan:
- Sarwar et al. (2001) - Item-based collaborative filtering
- Koren et al. (2009) - Matrix Factorization Techniques
- Borràs et al. (2014) - Tourism recommender systems

---

**Selamat mengerjakan paper Anda! 🎓✨**

Jika ada pertanyaan atau ingin memodifikasi dataset, silakan edit parameter di Cell 3:
- `n_users` untuk mengubah jumlah user
- `n_items` untuk mengubah jumlah destinasi
- `target_ratings` untuk mengubah jumlah rating